# 1 — Data

This project measures whether a fine-tuned language model learned a task or merely learned the *shape* of a task. To do that it needs two things: a set of poems, and a set of interpretations to train on.

Neither is an off-the-shelf dataset. The poems are fetched from a public API; the interpretations do not exist anywhere and are generated. This notebook introduces both sources, explains what is done to them, and reports what survived.

**All logic lives in `src/`.** This notebook imports, calls, and displays — nothing is implemented in a cell here. That is a deliberate constraint: it keeps the pipeline testable and means every number below comes from code that has a test next to it.

## The two sources

| | Source 1 — poems | Source 2 — interpretations |
|---|---|---|
| **Origin** | PoetryDB, a public HTTP API | Generated by a teacher model (DeepSeek API) |
| **Role** | model input, and the ground truth a quote is checked against | training target |
| **Exists already?** | yes, we retrieve it | no, we create it |
| **Licensing** | public domain | our own generations |
| **Trusted?** | yes — the poem text *is* the reference | **no — quality must be measured, not assumed** |

The asymmetry in the last row drives most of the design. A poem fetched from PoetryDB is simply correct: it is the artifact itself. A teacher-generated interpretation is a guess by a language model, and language models invent quotations. So the poems are cleaned, while the interpretations are cleaned **and audited** — see the hallucination rate below.

## Source 1 — PoetryDB

[PoetryDB](https://poetrydb.org) is a free HTTP API over a corpus of public-domain poetry. No key, no account, no rate-limit agreement to accept.

**Why this source.** Three properties made it the right testbed, and none of them are about poetry being interesting:

1. **Public domain** — the poem text can be printed in this repo, quoted in figures, and sent to third-party APIs without a copyright problem.
2. **Short** — a poem plus its interpretation fits comfortably in the model's context window. Prose fiction would not.
3. **Ungrounded output is detectable** — an interpretation is *supposed* to quote its source. That gives a mechanical, model-free check: did the quoted line actually appear in the poem? Most generation tasks have no such handle.

Poetry is the instrument here, not the subject. The evaluation would work on any task with a quotable source.

**How we retrieve it.** Two calls: fetch the author list, then fetch per author. Each record carries `title`, `author`, `lines` (a list of strings) and `linecount`. The API errors intermittently — on the order of a few percent of requests — so calls retry with backoff, and **every response is cached to disk on arrival**. The API is never hit twice for the same author.

**Selection rule.** Keep poems with `linecount` in **[8, 100]**. Below 8 lines there is too little text to interpret or to quote from. The upper bound is a compute and memory constraint, not an aesthetic judgement: Qwen2.5-0.5B has a 32K context, so the model itself is nowhere near the limit, but longer sequences cost quadratically more attention across 18 training runs on a fixed GPU budget.

Line count is only a **cheap pre-filter**. The binding constraint is a token-level check applied later, using the real tokeniser — see the filtering funnel below.

In [ ]:
import config
from src.data import fetch_poems

config.ensure_dirs()

# Printed so the settings a result was produced under are recorded next to it.
print(config.summary())

In [ ]:
# Reads the cache if present; only calls the API for authors not yet fetched.
raw_poems = fetch_poems.load_or_fetch()

print(f"{len(raw_poems)} poems retrieved\n")
fetch_poems.show_example(raw_poems[0])

## Source 2 — teacher-generated interpretations

There is no public dataset of poem interpretations, and hand-writing 2000 of them is not possible in the time available. So the training targets are **generated by a stronger model** (the teacher) and used to fine-tune a small one (the student). This is standard practice, and it has a standard failure mode that this project is built to detect.

**What this costs us, stated plainly.** The student's ceiling is the teacher's quality. If the teacher writes fluent interpretations that quote lines the poem does not contain, the student learns to do the same. Every claim this project makes is therefore about *grounding* — whether output is anchored to its input — and never about interpretation quality in an absolute sense. There is no expert reference to measure that against.

**One fixed prompt, never changed mid-run.** The prompt template lives in `config.py` and is printed verbatim below. If it changed partway through generation, the corpus would be a mixture of two tasks and every later comparison would be confounded. It is displayed here from `config` rather than retyped, so this notebook cannot drift from what actually ran.

**The required schema** — four parts, and part 2 is the one that matters:

1. Central idea
2. Two or three key images, **quoting the exact lines**
3. Tone
4. One specific interpretive claim

Part 2 is what makes grounding measurable. It forces the output to make a checkable claim about the source text.

In [ ]:
print(config.TEACHER_PROMPT_TEMPLATE)

**Generation is resumable, and at this scale that is not optional.** Roughly 2000 API calls will not complete in one uninterrupted run — the connection drops, a call times out, the process is stopped. So `generate.py` appends to JSONL, tracks which poem IDs are already done, skips them on restart, and logs failures instead of crashing. A crash halfway through costs nothing but the time already spent.

This is the longest single step in the project. Start it before anything else on day 2 and let it run while other work continues.

Temperature is held at 0.3–0.5: low enough for the schema to be followed reliably, high enough that every interpretation is not the same sentence.

In [ ]:
from src.data import generate

# Resumable: returns immediately for poems already present in the JSONL.
interpretations = generate.load_or_generate(raw_poems)

print(f"{len(interpretations)} interpretations available\n")
generate.show_example(interpretations[0])

## How the two are combined

A training example is one `(poem, interpretation)` pair. Getting from raw API responses to usable pairs is a filtering funnel, and **the count at every stage is recorded** — reported below as a table, because a corpus is not described by its final size alone. What was discarded, and why, says more.

A pair is dropped if:

| Rule | Reason |
|---|---|
| quoted lines absent from the poem | the teacher hallucinated — see below |
| interpretation outside [80, 250] words | too thin to be an interpretation, or padded |
| the four-part schema not followed | not the task we are training |
| poem + interpretation exceeds `MAX_SEQ_LEN` **in real tokens** | cannot be trained on whole |

Note that the first rule uses the poem as ground truth to audit the interpretation. This is the same substring check later used to score the student's own output, which means **the teacher is held to exactly the standard the student will be held to.**

**Nothing is ever truncated.** A pair that does not fit is dropped outright. Truncating would quietly corrupt the central measurement: the model would never see the tail of the poem, while the grounding checker still matches quotes against the full text. A quote from the cut region would then score as grounded when the model could not possibly have read it. Dropping is the only safe response, and the drop count appears in the funnel like any other.

The token check uses the real tokeniser rather than a line-count estimate, because the relationship between lines and tokens varies with vocabulary, punctuation, and archaic spelling — all common in this corpus.

In [ ]:
from src.data.filter import build_corpus, funnel_table

corpus, funnel = build_corpus(raw_poems, interpretations)
funnel_table(funnel)   # Figure 1 — data funnel

### Teacher hallucination rate

The share of teacher interpretations quoting at least one line that does not appear in its poem. This is reported, not hidden, for two reasons.

First, it is the honest characterisation of the training data — the targets are synthetic, and this is how good they are. Second, it sets expectations for the student: a model trained on targets that quote accurately *can* learn to quote accurately, but nothing forces it to. Whether it does is the question the rest of the project answers.

In [ ]:
from src.data.filter import hallucination_rate

rate, ci = hallucination_rate(interpretations)
print(f"teacher hallucination rate: {rate:.1%}  (95% CI {ci[0]:.1%}\u2013{ci[1]:.1%})")

## Splitting — 5-fold grouped cross-validation

The corpus is partitioned into **5 folds, grouped by author**. For each fold *k*, a model trains on the other four (~1600 poems, with ~10% of that held back as validation for loss curves) and fold *k* is held out.

**Grouped by author, not cut at the poem level.** This is the important choice. Poems by one author are not independent samples — themes, diction and preoccupations recur across a body of work. Training on one Dickinson poem therefore carries information about every other Dickinson poem. Splitting at the poem level would place correlated items on both sides of the partition, and the model could produce a plausible interpretation of a held-out Dickinson poem by drawing on what it learned from her other forty, without reading this one at all. That is the exact failure this project exists to detect, arriving through the back door.

This is standard **grouped cross-validation** — the same principle that keeps one patient's scans out of both train and test in medical imaging, or one speaker's recordings in speech recognition. `sklearn`'s `GroupKFold` implements it with the group set to the author.

**What grouping cannot fix.** Qwen was pretrained on web-scale text, and public-domain poetry is public domain *because* it is old and widely reproduced — so these poems, and published commentary about them, are very likely in the pretraining data already. Author priors exist in the model before fine-tuning touches it. Grouped folds prevent leakage through *our* training data; they do nothing about leakage that arrived during pretraining. Only the same-author swap condition catches both, because it asks whether the output is poem-specific regardless of where the knowledge came from.

**Why folds rather than a single 80/10/10 split.** The headline question is whether fine-tuning improves grounding. With one split, the answer could depend on which poems the model happened to train on, and there would be no way to tell. Five folds make that visible: the spread across fold means *is* the estimate of how much the conclusion depends on the training draw.

**Evaluation set: 30 poems sampled from each fold's held-out portion, 150 in total.** Not the full ~400 held out per fold, which would mean 2000 judged outputs and over thirteen times the judge API cost. Sampling 30 per fold **fixes the judge budget independently of corpus size** while guaranteeing that every evaluated poem was held out by the model that generated for it.

Fold sizes cannot be balanced exactly once authors are kept together, so the imbalance is checked here rather than discovered later as an unexplained result.

In [ ]:
from src.data import splits

# Exemplars are reserved FIRST so they can be kept out of every fold. Their
# AUTHORS are reserved too — a few-shot example by Dickinson would otherwise
# leak her themes into every prompt while she also sits in an evaluation fold.
exemplars = splits.reserve_exemplars(corpus, n=config.N_FEWSHOT, seed=config.SEED)

# Grouped by author: no author appears in more than one fold. Computed once,
# locally, and shipped to Kaggle as data — never recomputed there, since a
# different partition upstream would break the held-out guarantee silently.
folds = splits.make_folds(
    corpus,
    k=config.N_FOLDS,
    group_key=config.FOLD_GROUP_KEY,
    seed=config.SEED,
    exclude=exemplars,
)

# Evaluation poems are restricted to authors with a sibling poem available, so
# the same-author swap condition is computable for every one of them.
eval_set = splits.sample_eval_poems(
    folds,
    per_fold=config.EVAL_PER_FOLD,
    min_poems_per_author=config.MIN_POEMS_PER_AUTHOR_FOR_EVAL,
    seed=config.SEED,
)

splits.assert_partition_complete(folds, corpus, exclude=exemplars)
splits.assert_no_author_across_folds(folds)          # what makes it grouped CV
splits.assert_no_leakage(folds, eval_set)
splits.assert_exemplars_disjoint(eval_set, exemplars)
splits.assert_same_author_sibling_exists(eval_set, corpus)
splits.assert_balanced(folds, tolerance=config.MAX_FOLD_SIZE_IMBALANCE)

print(splits.summary(folds, eval_set))
print(f"\n{config.N_EVAL_POEMS} evaluation poems ✓")
print("no author spans two folds ✓")
print("every evaluation poem has a same-author sibling ✓")

## Corpus statistics and sampling bias

The figures below describe what the corpus actually is. They matter because every result in this project is conditional on this particular collection of poems.

**Author distribution is not decorative here.** It feeds the central measurement. The swap test scores each interpretation against a different poem *by the same author*, which is the strict control on whether the model read this poem or merely recognised its author. That condition only exists for authors with more than one poem in the corpus, so the shape of the author distribution determines how much of the corpus can carry the strict test at all.

**Known biases, stated up front:**

- **Public domain means old.** Copyright is why this corpus is usable at all, and it is also why the poems skew heavily pre-20th-century. The language is not contemporary English.
- **English only.**
- **A canon, not a sample.** PoetryDB collects widely-anthologised poets. It is not a random draw from poetry, and the authors represented are disproportionately male, British and American.
- **Uneven author coverage.** Poems arrive per author, so prolific anthologised poets contribute many more poems than others. The author distribution figure shows how skewed this is.
- **The evaluation set is biased toward prolific authors**, because evaluation poems are drawn only from authors with a sibling poem available. Worth noting which way this bias runs: prolific authors are exactly where author-prior leakage is strongest, so the evaluation set is *harder* than a random draw would be. The bias works against this project's own conclusions rather than flattering them.
- **Length-bounded to [8, 100] lines.** Epigrams and long-form work are outside the corpus. This bound is wider than a small-context model would have allowed — Qwen2.5-0.5B's 32K context is not the limiting factor, GPU budget is — but it is still a bound, and nothing here speaks to poems outside it.

None of these are fixable within this project, and none invalidate it — the question being asked is whether a fine-tuned model grounds its output in whatever text it is given. But they do bound the claim. Results describe this corpus, this teacher, and this task, and are not evidence about poetry interpretation in general.

In [ ]:
from src.plots import figures

figures.corpus_statistics(corpus)   # Figure 2 — line-count histogram, author distribution

## What carries forward

| Artifact | Used by |
|---|---|
| the corpus + **fold assignment** (`poem_id → fold_id`) | `02_training.ipynb` — uploaded to Kaggle; five `lora_r8` runs, one per fold |
| the 150 evaluation poems, tagged with their fold | generation for all five arms |
| funnel counts + hallucination rate | `04_report.ipynb` — data quality section |

Training runs on Kaggle; everything else, including all evaluation, runs locally. The next notebook covers that boundary.

Two things to carry forward in particular.

**The fold assignment is computed here and only here.** It travels to Kaggle as data. If it were recomputed there with a different seed, the held-out guarantee would break silently — nothing would raise, and every grounding number would quietly improve. When generating, each poem must be routed to the adapter of the fold that held it out.

**The teacher hallucination rate above is the reference point for reading every later grounding number.** A student that quotes as accurately as its teacher has learned what it was shown. A student that quotes less accurately has learned the format and dropped the substance — which is precisely the distinction this project exists to measure.